In [ ]:
# Imported libraries.
import requests
import importlib
from bs4 import BeautifulSoup
import pandas as pd
import snowflake.connector
import sqlalchemy as db
import numpy as np
import import_ipynb
import re
from datetime import datetime, timedelta

# Import my files.
import snowflake_functions as sf  # Import the notebook as a module.
import transform_data as td 
import extract_data as ed 
from skills_array import get_skills_array
skills_array = get_skills_array() # get full list of skills.

In [ ]:
# upload latest function
importlib.reload(sf)
importlib.reload(ed)
importlib.reload(td)

# connection snowflake
engine = sf.connect_snowflake()
connection = engine.connect()

# intialise tables that will later be loaded to snowflake
job_table = pd.DataFrame(columns=['JOB_ID', 'TITLE','COMPANY','LOCATION','EMPLOYMENT_TYPE','SALARY','PAY_PERIOD','POST_DATE'])
job_skills_table = pd.DataFrame(columns=["skill_id", "skill_name"])


# get the job_id of the last entry in the jobs table
job_id = sf.last_job_id(connection) + 1

# go through many pages
max_pages = 10
for j in range(max_pages):
    
    # get all job URL links
    full_urls = ed.get_URLs_to_jobs(j) 
    
    # go through all the URLs and extract relevant data.
    for k, url in enumerate(full_urls):   
        
        # Scrape job URL.
        response = requests.get(url) 
        soup = BeautifulSoup(response.content, "html.parser") 
        
        # Enter job ad in to jobs table data frame
        job_props, req_skills = ed.get_job_data(soup, skills_array) # extract the information from job
        parsed_salary = td.parse_salary(job_props[4]) # clean up salary   
        job_table = td.update_job_table(job_props, parsed_salary, job_table, job_id + k) # put into data frame.


        # get that entries id, get skill ids and enter it into job_skills table.
        df_skill_ids = sf.get_skill_ids(req_skills, connection)
        job_skills_table = td.update_job_skills_table(job_props, parsed_salary, job_skills_table, df_skill_ids, job_id + k)
        

# upload tables to snowflake database
job_table.to_sql('jobs', con=engine, if_exists='append', index=False)
job_skills_table.to_sql('job_skills', con=engine, if_exists='append', index=False)

# Close connection
connection.close()

Connected to Snowflake!
https://www.seek.com.au/job/82401270?type=standard&ref=search-standalone
https://www.seek.com.au/job/82379770?type=standard&ref=search-standalone
https://www.seek.com.au/job/82356435?type=standard&ref=search-standalone
https://www.seek.com.au/job/82321927?type=standard&ref=search-standalone
https://www.seek.com.au/job/82399006?type=standard&ref=search-standalone
https://www.seek.com.au/job/82294554?type=standard&ref=search-standalone
https://www.seek.com.au/job/82105910?type=standard&ref=search-standalone
https://www.seek.com.au/job/82297751?type=standard&ref=search-standalone
https://www.seek.com.au/job/82402882?type=standard&ref=search-standalone
https://www.seek.com.au/job/82257256?type=standard&ref=search-standalone
https://www.seek.com.au/job/82256211?type=standard&ref=search-standalone
https://www.seek.com.au/job/82305280?type=standard&ref=search-standalone
https://www.seek.com.au/job/82395274?type=standard&ref=search-standalone
https://www.seek.com.au/job

In [ ]:
# Close connection
connection.close()

In [ ]:
j

9